# Bearing RUL model development

This notebook is a lightweight companion to `scripts/train_model.py`. Reusable training and feature-extraction logic lives in Python modules so the API and training workflow do not drift apart.

The full training command is:

```bash
python -m scripts.train_model
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from bearing_app.features import extract_health_indicators
from scripts.train_model import (
    SCALE_CANDIDATES,
    load_signal_column,
    periodicity_score,
    select_wavelet_scale,
)


## Inspect one training run

The original project selected the first signal column from `2nd_test` as part of the training data. The files are loaded in filename order to preserve the recorded sequence.


In [ ]:
signals = load_signal_column("2nd_test", signal_column=0)
len(signals), signals[0].shape


## Select the Morlet CWT scale

Only the wavelet scale is searched in this implementation. The earlier notebook also referred to a beta parameter, but PyWavelets' built-in `morl` wavelet did not use that value in the project code.


In [ ]:
tuning_signal = signals[-100]
scores = [periodicity_score(tuning_signal, float(scale)) for scale in SCALE_CANDIDATES]
selected_scale = select_wavelet_scale(tuning_signal)
selected_scale


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(SCALE_CANDIDATES, scores, marker="o")
plt.xlabel("Wavelet scale")
plt.ylabel("SVD periodicity score")
plt.title("Wavelet-scale selection on training data")
plt.show()


## Health-indicator curve

Each vibration snapshot is reduced to the RMS of the real Morlet CWT coefficient at the selected scale. The XGBoost model receives windows of 15 consecutive HI values.


In [ ]:
health_indicator = extract_health_indicators(signals, selected_scale)

plt.figure(figsize=(10, 4))
plt.plot(health_indicator)
plt.xlabel("Recorded cycle")
plt.ylabel("Health indicator (RMS)")
plt.title("Health-indicator trend: 2nd_test, signal column 0")
plt.show()


## Full training and evaluation

Use the training module for the reproducible run. It trains on the selected columns from `2nd_test` and `3rd_test`, validates on the selected `1st_test` column, saves the model under `bearing_app/Assets/`, and writes metrics to `reports/metrics.json`.
